# 02 - Exploratory Data Analysis
## Diurnal & Seasonal Patterns in Generation, Load, and Market Prices

This notebook analyzes the historical relationship between solar PV generation, system electrical demand, and day-ahead electricity prices in Denmark West (`DK_1`).


In [1]:
import sys
sys.path.insert(0, '..')

from src.preprocessing import get_clean_data
import pandas as pd
import numpy as np

clean_df = get_clean_data()
clean_df['hour'] = clean_df.index.hour
clean_df['month'] = clean_df.index.month
clean_df['season'] = clean_df['month'].map({
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Autumn', 10: 'Autumn', 11: 'Autumn'
})


[2026-10-25 13:11:06] [INFO] [src.preprocessing]: Loading cached clean data from /home/asus/Projects/Renewable-Energy-Intelligence/data/processed/DK_1_clean.parquet


### 1. Diurnal Generation and Price Profiles by Season

In [2]:
diurnal = clean_df.groupby(['season', 'hour']).agg({
    'solar_generation_mw': 'mean',
    'load_actual_mw': 'mean',
    'price_day_ahead_eur_mwh': 'mean'
}).reset_index()

print("Average midday solar generation (Hour 12 UTC) by Season:")
print(diurnal[diurnal['hour'] == 12][['season', 'solar_generation_mw', 'price_day_ahead_eur_mwh']])


Average midday solar generation (Hour 12 UTC) by Season:
    season  solar_generation_mw  price_day_ahead_eur_mwh
12  Autumn           145.067947                34.975712
36  Spring           275.743591                25.185543
60  Summer           304.135345                30.131630
84  Winter            72.644852                32.502544


### 2. Spot Price Distribution & Negative Pricing Frequency

In [3]:
prices = clean_df['price_day_ahead_eur_mwh']
neg_hours = (prices < 0).sum()
pct_neg = (prices < 0).mean() * 100
print(f"Total negative price hours: {neg_hours:,} ({pct_neg:.2f}%)")
print(f"Minimum spot price: {prices.min():.2f} EUR/MWh")
print(f"Median spot price:  {prices.median():.2f} EUR/MWh")
print(f"99th percentile:    {prices.quantile(0.99):.2f} EUR/MWh")


Total negative price hours: 539 (1.07%)
Minimum spot price: -58.80 EUR/MWh
Median spot price:  30.29 EUR/MWh
99th percentile:    71.90 EUR/MWh
